# SDXL Convert Phase 4b-5 (TPU v5e-8, UNet quantizer + context-bin + package)

Consumes B1 output (CLIP+VAE already converted, UNet DLC from qairt-converter) via kernel_sources.
Runs only UNet qairt-quantizer (256 samples, ~3.5-4h) + qnn-context-binary-generator (~40min)
then packages final zip. Must have working TPU heartbeat (cell 10) to survive Kaggle 2h idle policy.


In [ ]:
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME         = '__MODEL_NAME__'
REALISTIC          = '__REALISTIC__' == 'true'
MIN_SOC            = '__MIN_SOC__'
BURN_TPU           = '__BURN_TPU__' == 'true'
print(f'civitai={CIVITAI_VERSION_ID} name={MODEL_NAME} realistic={REALISTIC} min_soc={MIN_SOC} burn_tpu={BURN_TPU}')

In [ ]:
!free -h
!nproc
!df -h /kaggle/working /tmp
!cat /etc/os-release | head -5
!ldd --version | head -1
!grep -m1 'model name' /proc/cpuinfo
!grep -m1 'cpu MHz' /proc/cpuinfo
!lscpu | grep -E 'Model name|Socket|Thread|Core|Vendor|Flags' | head -10

In [ ]:
# Locate B1 output via kernel_sources mount
import glob, os
candidates = glob.glob('/kaggle/input/**/b1_out', recursive=True)
if not candidates:
    # Try alternate patterns
    candidates = glob.glob('/kaggle/input/**/b1_meta.json', recursive=True)
    candidates = [os.path.dirname(c) for c in candidates]
assert candidates, 'B1 output (b1_out/) not found in /kaggle/input/ — was B1 kernel_sources mounted?'
B1_DIR = candidates[0]
print(f'B1 dir: {B1_DIR}')
!ls -lh {B1_DIR}
# Verify B1 completed
assert os.path.exists(f'{B1_DIR}/b1_meta.json'), 'b1_meta.json missing — B1 may not have completed'
import json as _json
_meta = _json.load(open(f'{B1_DIR}/b1_meta.json'))
assert _meta.get('b1_completed') is True, f'B1 did not complete: {_meta}'
print(f'B1 meta: {_meta}')

In [ ]:
import os
os.environ['MIN_SOC'] = MIN_SOC
os.environ['MODEL_NAME'] = MODEL_NAME

In [ ]:
# Install tools + convertsdxl + shell helper
import os, subprocess

def _run(cmd, log=None, check=True):
    """bash with errexit + pipefail; pipe to tee if log. Raises on non-zero."""
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        wrapped = f'set -eo pipefail; {cmd} 2>&1 | tee {log}'
    else:
        wrapped = f'set -eo pipefail; {cmd}'
    rc = subprocess.call(['bash', '-c', wrapped])
    if check and rc != 0:
        raise RuntimeError(f'shell failed (rc={rc}): {cmd[:200]}  log={log}')
    return rc

# Ensure apt cache is fresh before any installs
_run('apt-get update -y > /dev/null')
_run('apt-get install -y unzip zip curl time > /dev/null')
# libc++/libc++abi/libunwind needed by QNN SDK libPyIrGraph.so / libQnnHtp.so — Kaggle Debian 13 trixie doesn't ship them
# Specific LLVM runtime packages — generic libc++-dev/libunwind-dev do NOT provide libunwind.so.1 (GNU libunwind ships .so.8)
_run('apt-get install -y libc++1-19 libc++abi1-19 libunwind-19 > /dev/null')
_run('ldconfig')  # refresh ld cache so newly installed libs are found by ldd / dlopen
_run('curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1')
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

_run('rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip')
_run("curl -sL --fail --retry 5 --retry-delay 5 -o /tmp/convertsdxl.zip 'https://chino.icu/local-dream/convertsdxl.zip'")
_run('mkdir -p /tmp/convertsdxl_unzip')
_run('unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip')

root = subprocess.check_output(
    "find /tmp/convertsdxl_unzip -maxdepth 3 -name 'export_sdxl.sh' -printf '%h\\n' | head -1",
    shell=True, text=True).strip()
assert root, 'export_sdxl.sh not found in unzipped convertsdxl'
NPUCONVERT_DIR = os.path.abspath(root)
os.environ['NPUCONVERT_DIR'] = NPUCONVERT_DIR
print(f'NPUCONVERT_DIR: {NPUCONVERT_DIR}')

In [ ]:
# QNN SDK
import os, subprocess
_run('mkdir -p /tmp/qnn_sdk')
_run(
    "curl -sL --fail --retry 5 --retry-delay 5 -A 'Mozilla/5.0' "
    "-o /tmp/qnn_sdk/qnn.zip "
    "'https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip'"
)
_run('cd /tmp/qnn_sdk && unzip -q qnn.zip')
bin_file = subprocess.check_output(
    'find /tmp/qnn_sdk -type f -name envsetup.sh -print -quit',
    shell=True, text=True).strip()
assert bin_file, 'envsetup.sh not found in QNN SDK extract'
QNN_SDK_BIN = os.path.dirname(bin_file)
QNN_SDK_ROOT_DIR = os.path.dirname(QNN_SDK_BIN)
os.environ['QNN_SDK_BIN'] = QNN_SDK_BIN
os.environ['QNN_SDK_ROOT_DIR'] = QNN_SDK_ROOT_DIR
print(f'QNN_SDK_ROOT: {QNN_SDK_ROOT_DIR}')

In [ ]:
# B2 does not need model.safetensors — UNet DLC already in B1 output.
# Verify all required B1 artifacts are present before continuing.
import os
SOC = MIN_SOC
required = [
    f'{B1_DIR}/unet_sdxl/model.dlc',
    f'{B1_DIR}/input_list_unet_sdxl.txt',
    f'{B1_DIR}/unet_input_raw_sdxl',
    f'{B1_DIR}/converted',
    f'{B1_DIR}/htp_backend_{SOC}.json',
    f'{B1_DIR}/htp_config_{SOC}.json',
]
missing = [p for p in required if not os.path.exists(p)]
assert not missing, f'B1 artifacts missing (B1 did not reach cell 16?):\n' + '\n'.join(missing)
print(f'all {len(required)} B1 artifacts present')
import os
os.environ['B1_DIR'] = B1_DIR

In [ ]:
# Python env
import os
os.chdir(os.environ['NPUCONVERT_DIR'])
_run('uv venv -p 3.10.17 --clear')
_run('. .venv/bin/activate && uv sync')

In [ ]:
# Copy B1 artifacts into NPUCONVERT_DIR (where qairt-quantizer / qnn-context-binary-generator expect them)
import os, shutil
NPU = os.environ['NPUCONVERT_DIR']
SOC = os.environ['MIN_SOC']

# UNet model.dlc → NPU/unet_sdxl/model.dlc
os.makedirs(f'{NPU}/unet_sdxl', exist_ok=True)
shutil.copy(f'{B1_DIR}/unet_sdxl/model.dlc', f'{NPU}/unet_sdxl/model.dlc')

# UNet calibration data
shutil.copy(f'{B1_DIR}/input_list_unet_sdxl.txt', f'{NPU}/input_list_unet_sdxl.txt')
shutil.copytree(f'{B1_DIR}/unet_input_raw_sdxl', f'{NPU}/unet_input_raw_sdxl', dirs_exist_ok=True)

# Configs (qnn-context-binary-generator uses ./htp_backend_{SOC}.json from cwd)
shutil.copy(f'{B1_DIR}/htp_backend_{SOC}.json', f'{NPU}/htp_backend_{SOC}.json')
shutil.copy(f'{B1_DIR}/htp_config_{SOC}.json', f'{NPU}/htp_config_{SOC}.json')

# Pre-converted artifacts (clip/clip2/vae_enc/vae_dec .bin/.mnn) — B2 needs them to build final zip
os.makedirs(f'{NPU}/output', exist_ok=True)
if os.path.exists(f'{NPU}/output/qnn_models_sdxl_{SOC}'):
    shutil.rmtree(f'{NPU}/output/qnn_models_sdxl_{SOC}')
shutil.copytree(f'{B1_DIR}/converted', f'{NPU}/output/qnn_models_sdxl_{SOC}')

_run(f'ls -lh {NPU}/unet_sdxl/ {NPU}/output/qnn_models_sdxl_{SOC}/')
_run(f'du -sh {NPU}/unet_sdxl {NPU}/unet_input_raw_sdxl {NPU}/output')

In [ ]:
# Mem watcher + TPU heartbeat (gated by BURN_TPU flag set in cell 1).
# Kaggle policy kills TPU VMs idle > 2h. For >2h notebooks (B2 UNet quantizer 4h) this is required.
# For <2h notebooks (B1 ~1h48m) can set burn_tpu=false to skip (saves TF-TPU install + some CPU).
import subprocess, os, time, threading
os.makedirs('/kaggle/working/logs', exist_ok=True)

# --- mem watcher always runs (cheap, writes csv) ---
watcher_sh = '''#!/bin/bash
echo "epoch,datetime,MemAvail_MB,TopRSS_MB,TopProc"
while true; do
  epoch=$(date +%s); dt=$(date -Iseconds)
  mem=$(awk '/MemAvailable:/{print int($2/1024)}' /proc/meminfo)
  line=$(ps -eo rss,comm --sort=-rss --no-headers 2>/dev/null | head -1)
  rss=$(echo "$line" | awk '{print int($1/1024)}')
  proc=$(echo "$line" | awk '{print $2}')
  echo "$epoch,$dt,$mem,$rss,$proc"
  sleep 10
done
'''
open('/tmp/mem_watch.sh','w').write(watcher_sh)
os.chmod('/tmp/mem_watch.sh', 0o755)
p = subprocess.Popen(['/tmp/mem_watch.sh'], stdout=open('/kaggle/working/logs/mem-trace.csv','w'), stderr=subprocess.DEVNULL)
print(f'mem-watcher pid={p.pid}')

if not BURN_TPU:
    print('BURN_TPU=false → skipping TPU heartbeat install+thread. Kaggle may kill this kernel at 2h TPU-idle.')
else:
    # Install TF-TPU (Kaggle official v5e-8 stack) — uninstall default jax first
    _run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax 2>&1 | tail -3', check=False)
    _run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system --quiet '
         'tensorflow-tpu==2.18.0 '
         '--find-links https://storage.googleapis.com/libtpu-tf-releases/index.html')

    import tensorflow as tf
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    NR = strategy.num_replicas_in_sync
    print(f'Jupyter kernel TPU replicas: {NR}')
    assert NR > 0

    with strategy.scope():
        _model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(32,32,3)),
            tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
            tf.keras.layers.MaxPool2D(),
            tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(10),
        ])
        _opt = tf.keras.optimizers.Adam(1e-3)

    BATCH_PER_REPLICA = 128
    GLOBAL_BATCH = BATCH_PER_REPLICA * NR

    @tf.function
    def _train_step(x, y):
        with tf.GradientTape() as tape:
            logits = _model(x, training=True)
            loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(y, logits))
        grads = tape.gradient(loss, _model.trainable_variables)
        _opt.apply_gradients(zip(grads, _model.trainable_variables))
        return loss

    def _dataset():
        def _gen():
            while True:
                x = tf.random.normal((GLOBAL_BATCH, 32, 32, 3))
                y = tf.random.uniform((GLOBAL_BATCH,), maxval=10, dtype=tf.int64)
                yield x, y
        ds = tf.data.Dataset.from_generator(
            _gen,
            output_signature=(
                tf.TensorSpec(shape=(GLOBAL_BATCH,32,32,3), dtype=tf.float32),
                tf.TensorSpec(shape=(GLOBAL_BATCH,), dtype=tf.int64),
            ))
        return strategy.experimental_distribute_dataset(ds)

    _stop = threading.Event()
    _step_count = [0]
    def _hb_loop():
        ds = _dataset()
        it = iter(ds)
        t_last_log = time.time()
        while not _stop.is_set():
            try:
                x, y = next(it)
                per_replica = strategy.run(_train_step, args=(x, y))
                loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica, axis=None)
                _ = float(loss)
                _step_count[0] += 1
                if time.time() - t_last_log >= 60:
                    with open('/kaggle/working/logs/tpu_heartbeat.log', 'a') as lf:
                        lf.write(f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] steps={_step_count[0]} loss={float(loss):.4f} replicas={NR}\n')
                        lf.flush()
                    t_last_log = time.time()
            except Exception as e:
                with open('/kaggle/working/logs/tpu_heartbeat.log', 'a') as lf:
                    lf.write(f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] ERROR: {e!r}\n')
                time.sleep(5)

    _hb_thread = threading.Thread(target=_hb_loop, daemon=True, name='tpu-keepalive')
    _hb_thread.start()
    print(f'tpu-keepalive THREAD ident={_hb_thread.ident}')
    time.sleep(45)
    assert _hb_thread.is_alive(), 'keepalive thread died'
    assert _step_count[0] >= 5, f'keepalive only did {_step_count[0]} steps in 45s'
    print(f'keepalive confirmed: {_step_count[0]} steps in 45s, thread alive')
    _run('tail -3 /kaggle/working/logs/tpu_heartbeat.log', check=False)

In [ ]:
# Phase 2 skipped in B2 (gen_quant_data_sdxl done in B1)
print('Phase 2 skipped — B2 reuses B1 calibration data')

In [ ]:
# Phase 3 skipped in B2 (export_onnx done in B1, UNet DLC via B1 output)
print('Phase 3 skipped — B2 consumes B1 UNet DLC directly')

In [ ]:
# phase3_backup skipped (B1 already saved b1_out equivalent)
print('phase3_backup skipped — B1 b1_out supersedes it')

In [ ]:
# Absolute-path fix for htp_backend json (defensive, same as B1 cell 14 pattern)
import os, json as _json
NPU = os.environ['NPUCONVERT_DIR']
SOC = os.environ['MIN_SOC']
hbp = f'{NPU}/htp_backend_{SOC}.json'
d = _json.load(open(hbp))
d['backend_extensions']['config_file_path'] = f'{NPU}/htp_config_{SOC}.json'
_json.dump(d, open(hbp, 'w'), indent=2)
print('patched htp_backend json:', open(hbp).read())

In [ ]:
# B2 Phase 4b-5: UNet qairt-quantizer + qnn-context-binary-generator.
# Flags match upstream scripts/convert_unet_sdxl.sh EXACTLY (including upstream's double --output_dir).
import time, os, subprocess, shutil
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
SOC = os.environ['MIN_SOC']
NPU = os.environ['NPUCONVERT_DIR']
QNN = os.environ['QNN_SDK_ROOT_DIR']

# libpython LD path (same fix as B1)
_pylib = subprocess.check_output(
    ['.venv/bin/python', '-c', 'import sys,os; print(os.path.join(sys.base_prefix, "lib"))'],
    text=True).strip()
assert os.path.exists(os.path.join(_pylib, 'libpython3.10.so.1.0'))
os.environ['LD_LIBRARY_PATH'] = f"{_pylib}:{os.environ.get('LD_LIBRARY_PATH','')}"

# ldd pre-flight (same as B1)
_qnn_lib = f'{QNN}/lib/x86_64-linux-clang'
_check_env = os.environ.copy()
_check_env['LD_LIBRARY_PATH'] = f'{_qnn_lib}:{_check_env["LD_LIBRARY_PATH"]}'
for _lib in [
    f'{QNN}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrQuantizer.so',
    f'{QNN}/lib/x86_64-linux-clang/libQnnHtp.so',
    f'{QNN}/lib/x86_64-linux-clang/libQnnModelDlc.so',
]:
    _ldd = subprocess.check_output(['ldd', _lib], env=_check_env, text=True)
    _missing = [l.strip() for l in _ldd.splitlines() if 'not found' in l]
    assert not _missing, f'{_lib} unresolved:\n' + '\n'.join(_missing)
    print(f'OK  {os.path.basename(_lib)}')

# Required B1 artifacts check
for p in [f'{NPU}/unet_sdxl/model.dlc', f'{NPU}/input_list_unet_sdxl.txt',
          f'{NPU}/unet_input_raw_sdxl', f'{NPU}/htp_backend_{SOC}.json', f'{NPU}/htp_config_{SOC}.json']:
    assert os.path.exists(p), f'{p} missing'
print('all B2 required artifacts present')

# --- STAGE 1: UNet qairt-quantizer (3.5-4h on CPU) ---
quant_sh = f'''#!/bin/bash
set -eo pipefail
cd "{NPU}"
. .venv/bin/activate
source "{QNN}/bin/envsetup.sh"
cd "{NPU}"
export SUFFIX=_{SOC}
echo "=== B2 STAGE 1: UNet qairt-quantizer (256 samples, expect 3.5-4h) ==="
qairt-quantizer --input_dlc ./unet_sdxl/model.dlc --output_dlc ./unet_sdxl/model_quantized.dlc --act_bitwidth 16 --bias_bitwidth 32 --use_per_channel_quantization --input_list ./input_list_unet_sdxl.txt --preserve_io_datatype
echo "=== B2 STAGE 1 done ==="
'''
open('/tmp/b2_quant.sh', 'w').write(quant_sh)
os.chmod('/tmp/b2_quant.sh', 0o755)
_run(
    """/usr/bin/time -v bash /tmp/b2_quant.sh 2>&1 | tee /kaggle/working/logs/phase45_b2_quant.log | awk '!/Failed to set thread affinity for cpuset/'""",
    log=None
)
t_quant = int(time.time() - t0)
print(f'UNet quantizer elapsed: {t_quant}s')

# STAGE 1 checkpoint: copy quantized DLC to /kaggle/working EARLY so if context-bin fails in STAGE 2
# the expensive quantizer output is preserved (can be retried in a separate B3 kernel).
os.makedirs('/kaggle/working/b2_intermediate', exist_ok=True)
shutil.copy(f'{NPU}/unet_sdxl/model_quantized.dlc', '/kaggle/working/b2_intermediate/unet_model_quantized.dlc')
shutil.copy(f'{NPU}/htp_backend_{SOC}.json', f'/kaggle/working/b2_intermediate/htp_backend_{SOC}.json')
shutil.copy(f'{NPU}/htp_config_{SOC}.json', f'/kaggle/working/b2_intermediate/htp_config_{SOC}.json')
_run('ls -lh /kaggle/working/b2_intermediate/')
print(f'checkpoint: quantized DLC saved to /kaggle/working/b2_intermediate/ (usable if STAGE 2 fails)')

# --- STAGE 2: UNet qnn-context-binary-generator (upstream double --output_dir preserved) ---
ctxbin_sh = f'''#!/bin/bash
set -eo pipefail
cd "{NPU}"
. .venv/bin/activate
source "{QNN}/bin/envsetup.sh"
cd "{NPU}"
export SUFFIX=_{SOC}
echo "=== B2 STAGE 2: UNet qnn-context-binary-generator ==="
qnn-context-binary-generator --dlc_path ./unet_sdxl/model_quantized.dlc --model "{QNN}/lib/x86_64-linux-clang/libQnnModelDlc.so" --backend "{QNN}/lib/x86_64-linux-clang/libQnnHtp.so" --output_dir --output_dir "./output/qnn_models_sdxl${{SUFFIX}}" --binary_file unet --config_file ./htp_backend${{SUFFIX}}.json
echo "=== B2 STAGE 2 done ==="
'''
open('/tmp/b2_ctxbin.sh', 'w').write(ctxbin_sh)
os.chmod('/tmp/b2_ctxbin.sh', 0o755)
_run(
    """/usr/bin/time -v bash /tmp/b2_ctxbin.sh 2>&1 | tee /kaggle/working/logs/phase45_b2_ctxbin.log | awk '!/Failed to set thread affinity for cpuset/'""",
    log=None
)
print(f'B2 total elapsed: {int(time.time()-t0)}s')
_run('free -h')

In [ ]:
# Final package: B1's converted .bin/.mnn + UNet.bin just generated → single zip for app
import os, shutil
os.chdir(os.environ['NPUCONVERT_DIR'])
SOC = os.environ['MIN_SOC']
out_dir = f'output/qnn_models_sdxl_{SOC}'
assert os.path.isdir(out_dir)
# Verify UNet just produced
assert os.path.exists(f'{out_dir}/unet.bin'), 'unet.bin missing — qnn-context-binary-generator failed'
# Verify B1's work present (copied in cell 9)
for fname in ['clip.mnn', 'clip_2.mnn', 'vae_encoder.bin', 'vae_decoder.bin', 'tokenizer.json']:
    assert os.path.exists(f'{out_dir}/{fname}'), f'{fname} missing (B1 output copy may have been incomplete)'

_run(f'touch {out_dir}/SDXL')
zip_name = f'{os.environ["MODEL_NAME"]}_qnn2.28_{SOC}.zip'
_run(f'zip -r /kaggle/working/{zip_name} {out_dir}')
_run(f'ls -lh /kaggle/working/{zip_name}')
_run('df -h /kaggle/working')

# stop mem-watcher subprocess (best-effort)
try:
    _p = open('/tmp/watcher.pid').read().strip()
    if _p: os.kill(int(_p), 9)
except Exception as e:
    print(f'watcher kill skipped: {e}')

In [ ]:
# B2 Summary
import pandas as pd, os
if os.path.exists('/kaggle/working/logs/mem-trace.csv'):
    df = pd.read_csv('/kaggle/working/logs/mem-trace.csv')
    print(f'mem samples: {len(df)}')
    if len(df):
        print('peak memory (lowest MemAvail):')
        print(df.nsmallest(3, 'MemAvail_MB')[['datetime','MemAvail_MB','TopRSS_MB','TopProc']].to_string())
_run('tail -10 /kaggle/working/logs/tpu_heartbeat.log', check=False)
_run('ls -lh /kaggle/working/')